In [2]:
import torch
from common.model_poseformer import PoseTransformer

/home/gokul/ideaslab/ai-research-forcePose-fork/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/gokul/ideaslab/ai-research-forcePose-fork/.venv/lib/python3.12/site-packages/timm/models/helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/home/gokul/ideaslab/ai-research-forcePose-fork/.venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/gokul/ideaslab/ai-research-forcePose-fork/.venv/lib/python3.12/site-packages/t

In [3]:
model = PoseTransformer(
    num_frame=81,
    num_joints=17,
    in_chans=2,
    embed_dim_ratio=32,
    depth=4,
    num_heads=8,
    mlp_ratio=2.0,
    qkv_bias=True,
    qk_scale=None,
    drop_path_rate=0,
    pred_force=True
)

In [4]:
checkpoint = torch.load("checkpoint/coco_2d/2d_81frames_t2.bin", map_location=lambda storage, loc: storage, weights_only=False)
model.load_state_dict(checkpoint["model_pos"], strict=False)
model.eval()

PoseTransformer(
  (Spatial_patch_to_embedding): Linear(in_features=2, out_features=32, bias=True)
  (pos_drop): Dropout(p=0.0, inplace=False)
  (Spatial_blocks): ModuleList(
    (0-3): 4 x Block(
      (norm1): LayerNorm((32,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=32, out_features=96, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=32, out_features=32, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((32,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=32, out_features=64, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=64, out_features=32, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (blocks): ModuleList(
    (0-3): 4 x Block(
      (norm1): LayerNorm((544,), eps=1e-06, elementwise_affine=True)


In [5]:
dummy_input = torch.randn(1, 81, 17, 2).float()

traced_model = torch.jit.trace(model, dummy_input, strict=False)
traced_model.save("checkpoint/forcepose.pt")

In [6]:
import coremltools as ct

mlmodel = ct.convert(traced_model,
                     source="pytorch",
                     inputs=[ct.TensorType(name="input_1", shape=(1, 81, 17, 2))],
                     convert_to="mlprogram",
                     minimum_deployment_target=ct.target.macOS13,
                     debug=True)

mlmodel.save("checkpoint/forcepose.mlpackage")

Running MIL default pipeline:   0%|          | 0/89 [00:00<?, ? passes/s]/home/gokul/ideaslab/ai-research-forcePose-fork/.venv/lib/python3.12/site-packages/coremltools/converters/mil/mil/passes/defs/preprocess.py:273: UserWarning: Output, '677', of the source model, has been renamed to 'var_677' in the Core ML model.
  warnings.warn(msg.format(var.name, new_name))
Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 474.97 passes/s]
